# SAE basics: encode prompts and inspect features

A **Sparse Autoencoder (SAE)** takes a language model's internal activations and breaks them into a long list of features. Each feature roughly corresponds to one concept (a name, a topic, a grammatical role), and for any given input only a small fraction are active, which makes the model's internal state easier to read.

This notebook walks through the simplest SAE workflow with Murano:

1. Load a pre-trained SAE from HuggingFace.
2. Encode a few prompts through it.
3. For each prompt, see which feature fires strongest and what concept it represents.

We use Gemma 2 2B with the gemma-scope SAE at layer 20.

**Install**: `pip install -e ".[sae,notebook]"`

## Setup

Pick the model, the SAE release, and the SAE id. `SAEEncode` reads the target layer from the SAE's own config, so we don't have to specify it.

In [1]:
from murano import MuranoModel, Pipeline
from murano.steps import SAEEncode, SAEFeatureLabel, top_sae_features_per_prompt
from murano.steps.prompts import LoadPrompts

MODEL_ID = "google/gemma-2-2b-it"
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = "layer_20/width_16k/canonical"

model = MuranoModel(MODEL_ID)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Encode prompts through the SAE

A tiny pipeline: load prompts, then encode them through the SAE. We use five sentences, each about a landmark in a different country.

In [2]:
PROMPTS = [
    "The Eiffel Tower is located in the city of Paris",
    "The Colosseum is located in the city of Rome",
    "The Brandenburg Gate is located in the city of Berlin",
    "Senso-ji temple is located in the city of Tokyo",
    "Red Square is located in the city of Moscow",
]

results = Pipeline(
    [
        LoadPrompts(PROMPTS),
        SAEEncode(model, release=SAE_RELEASE, sae_id=SAE_ID),
    ]
).run()
record = results["sae_record"]

n_prompts, n_tokens, n_features = record.activations.shape
print(f"Encoded {n_prompts} prompts at layer {record.hook.layer}.")
print(f"Every token is now a {n_features}-feature SAE vector.")
print(
    f"activations shape = {tuple(record.activations.shape)}  [prompts, tokens, features]"
)

Encoded 5 prompts at layer 20.
Every token is now a 16384-feature SAE vector.
activations shape = (5, 13, 16384)  [prompts, tokens, features]


## SAEs are sparse

Even though the SAE has thousands of features, only a small fraction fire on any given token, so each token's feature vector is mostly zeros. Below we count how many fire per real (non-padding) token.

In [3]:
active_per_token = (record.activations > 0).sum(dim=-1)
mean_active = active_per_token[record.attention_mask.bool()].float().mean().item()
print(
    f"On average {mean_active:.0f} of {n_features} features fire per token "
    f"({100 * mean_active / n_features:.1f}% active)."
)

On average 751 of 16384 features fire per token (4.6% active).


## What is each sentence about?

To read what a sentence is about *as a whole*, we average each feature's activation across its content tokens and take the top one. `top_sae_features_per_prompt` does this and automatically drops the few high-frequency "broad" features that fire on almost every token (and would otherwise dominate the average). The result is one feature id per prompt; we learn what it means in the next cell.

In [4]:
top_feats = top_sae_features_per_prompt(record, n=1, reduce="mean")
for prompt, feats in zip(record.texts, top_feats):
    print(f"{prompt}")
    print(f"    -> top feature #{feats[0]}\n")

The Eiffel Tower is located in the city of Paris
    -> top feature #5516

The Colosseum is located in the city of Rome
    -> top feature #1780

The Brandenburg Gate is located in the city of Berlin
    -> top feature #16173

Senso-ji temple is located in the city of Tokyo
    -> top feature #1315

Red Square is located in the city of Moscow
    -> top feature #1116



## What does each feature mean?

A feature id is just a number. To learn what it represents, `SAEFeatureLabel` reads the feature's direction and projects it through the model's unembedding (the layer that turns hidden states into output-token logits). The token it pushes hardest is the one this feature *promotes*.

Each prompt is about a landmark in a different country, and each top feature promotes that country's name or nationality. That is the SAE doing its job: it has decomposed each sentence into a clean, interpretable concept.

In [5]:
feat_ids = sorted({fid for feats in top_feats for fid in feats})
results = Pipeline([SAEFeatureLabel(model, feat_ids=feat_ids, k_tokens=1)]).run(results)
labels = results["feature_labels"]

for prompt, feats in zip(record.texts, top_feats):
    fid = feats[0]
    promoted = labels.tokens[fid][0].strip()
    print(f"{prompt}")
    print(f"    -> feature #{fid} means {promoted!r}\n")

The Eiffel Tower is located in the city of Paris
    -> feature #5516 means 'French'

The Colosseum is located in the city of Rome
    -> feature #1780 means 'Italian'

The Brandenburg Gate is located in the city of Berlin
    -> feature #16173 means 'German'

Senso-ji temple is located in the city of Tokyo
    -> feature #1315 means 'Japan'

Red Square is located in the city of Moscow
    -> feature #1116 means 'Russian'



## What's next

Notebook [`02_feature_steering.ipynb`](02_feature_steering.ipynb) shows how to find a feature for a specific concept and steer the model's generations with it.